In [6]:
%load_ext autoreload
%autoreload 2
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import os
import argparse
import os
import sys
import numpy as np
from tqdm import tqdm
import torch
from matplotlib import pyplot as plt

sys.path.append(os.path.abspath('../src'))
from utils.data_utils import split_dataset, split_indices
from dataset.placenta import PlacentaDatasetHypergraph
from dataset.extend import ExtendedDataset
from dataset.mibi import MIBIDataset, MIBISubsetHypergraph
from utils.seed import seed_everything
from models.hypergraph_scattering import HypergraphScatteringNet
from train import prepare_dataloaders

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
cell_protein_df = pd.read_csv('../data/MIBI/raw/cell_protein_data.csv')
patient_df = pd.read_csv('../data/MIBI/raw/patient_info.csv')

In [9]:
from argparse import Namespace

args = Namespace(
    dataset='mibi',
    data_folder='../data/MIBI/patchified_all_genes',
    train_val_test_ratio='6:2:2',
    desired_batch_size=16,
    batch_size=1,
    k_hop=1,
    num_workers=4,
    max_epochs=50,
    max_training_iters=512,
    max_validation_iters=256,
    learning_rate=1e-2,
    random_seed=1,
)

In [52]:

@torch.no_grad()
def save_test_set_attentions(model, test_loader, device, attention_save_path):
    niche_attention_list = []
    feature_attention_arr, y_true_arr, y_pred_arr = None, None, None

    # Aggregate the MLP weights.
    linear_layers = [m for m in model.classifier.modules() if isinstance(m, torch.nn.Linear)]
    W = linear_layers[-1].weight
    for layer in linear_layers[:-1][::-1]:
        W = W @ layer.weight  # Chain multiplication
    mlp_weights = W.squeeze()  # [num_classes, wavelet_scales * num_features]

    for data_item in tqdm(test_loader):
        data_item = data_item.to(device)
        y_true = torch.Tensor(data_item.y).long().to(device)
        niche_attn, feature_attn = model(
            x=data_item.x,
            hyperedge_index=data_item.edge_index,
            hyperedge_attr=data_item.edge_attr,
            batch=data_item.batch,
            return_attention=True)
        y_pred = model(
            x=data_item.x,
            hyperedge_index=data_item.edge_index,
            hyperedge_attr=data_item.edge_attr,
            batch=data_item.batch)

        assert len(niche_attn.shape) == 1
        niche_attn = niche_attn.cpu().detach().numpy().astype(np.float16)
        feature_attn = feature_attn.cpu().detach().numpy().astype(np.float16)
        y_true = y_true.cpu().detach().numpy().reshape(1, 1)
        y_pred = y_pred.cpu().detach().numpy().reshape(1, -1)

        if feature_attention_arr is None:
            niche_attention_list = [niche_attn]
            feature_attention_arr = feature_attn
            y_true_arr = y_true
            y_pred_arr = y_pred
        else:
            niche_attention_list.append(niche_attn)
            feature_attention_arr = np.concatenate((feature_attention_arr, feature_attn), axis=0)
            y_true_arr = np.concatenate((y_true_arr, y_true), axis=0)
            y_pred_arr = np.concatenate((y_pred_arr, y_pred), axis=0)

    with open(attention_save_path, 'wb+') as f:
        np.savez(
            f,
            niche_attention_arr=np.array(
                [attn.cpu().numpy() if hasattr(attn, "cpu") else attn
                for attn in niche_attention_list],
                dtype=object
            ),
            feature_attention_arr=feature_attention_arr.cpu().numpy() if hasattr(feature_attention_arr, "cpu") else feature_attention_arr,
            y_true_arr=y_true_arr.cpu().numpy() if hasattr(y_true_arr, "cpu") else y_true_arr,
            y_pred_arr=y_pred_arr.cpu().numpy() if hasattr(y_pred_arr, "cpu") else y_pred_arr,
            mlp_weights=mlp_weights.cpu().numpy() if hasattr(mlp_weights, "cpu") else mlp_weights
        )
    return


def visualize_test_set_attention(embedding_save_path, gene_list, class_map):
    npzfile = np.load(embedding_save_path, allow_pickle=True)
    niche_attention_arr = npzfile['niche_attention_arr']
    feature_attention_arr = npzfile['feature_attention_arr']
    y_true_arr = npzfile['y_true_arr']
    y_pred_arr = npzfile['y_pred_arr']
    mlp_weights = npzfile['mlp_weights']

    # Softmax over classes.
    y_pred_arr = softmax(y_pred_arr)
    confidence_arr = y_pred_arr.max(axis=1).reshape(-1, 1)
    y_pred_binary_arr = y_pred_arr.argmax(axis=1).reshape(-1, 1)

    fig = plt.figure(figsize=(24, 16))
    for class_idx, class_name in class_map.items():
        subject_indices = (y_true_arr == class_idx).flatten()

        attention_curr_class = feature_attention_arr[subject_indices, ...].mean(axis=0)
        attention_curr_class = (attention_curr_class - attention_curr_class.min()) / (
            attention_curr_class.max() - attention_curr_class.min())

        ax = fig.add_subplot(2, len(class_map.items()), class_idx + 1)
        matrix_fig = ax.imshow(attention_curr_class, cmap='coolwarm')
        ax.set_title(class_name, fontsize=16)
        cbar = fig.colorbar(matrix_fig, ax=ax)
        ticks = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
        cbar.set_ticks(ticks)
        cbar.set_ticklabels(ticks)

        if class_idx == 0:
            ax.set_ylabel('Attention weights only', fontsize=16)

    fig.tight_layout(pad=2)
    fig.savefig(os.path.join(os.path.dirname(embedding_save_path), 'feature_attentions.png'))

    for class_idx, class_name in class_map.items():
        subject_indices = (y_true_arr == class_idx).flatten()

        attention_curr_class = feature_attention_arr[subject_indices, ...].mean(axis=0)
        mlp_weight_curr_class = mlp_weights[class_idx, :]
        attention_curr_class = attention_curr_class * mlp_weight_curr_class[None, :]
        attention_curr_class = (attention_curr_class - attention_curr_class.min()) / (
            attention_curr_class.max() - attention_curr_class.min())

        ax = fig.add_subplot(2, len(class_map.items()), len(class_map.items()) + class_idx + 1)
        matrix_fig = ax.imshow(attention_curr_class, cmap='coolwarm')
        ax.set_title(class_name, fontsize=16)
        cbar = fig.colorbar(matrix_fig, ax=ax)
        ticks = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
        cbar.set_ticks(ticks)
        cbar.set_ticklabels(ticks)

        if class_idx == 0:
            ax.set_ylabel('Attention weights scaled by MLP weights', fontsize=16)

    fig.tight_layout(pad=2)
    fig.savefig(os.path.join(os.path.dirname(embedding_save_path), 'feature_attentions.png'))
    plt.close(fig)

    num_features = feature_attention_arr.shape[-1]
    assert num_features == len(gene_list)
    fig = plt.figure(figsize=(32, num_features//2))
    colors = plt.cm.Blues(np.linspace(0.2, 0.8, num_features))
    for class_idx, class_name in class_map.items():
        subject_indices = (y_true_arr == class_idx).flatten()

        attention_curr_class = feature_attention_arr[subject_indices, ...].mean(axis=0)
        attention_curr_class = (attention_curr_class - attention_curr_class.min()) / (
            attention_curr_class.max() - attention_curr_class.min())

        feature_importance = attention_curr_class.sum(axis=0)
        # Sort descendingly.
        sorted_idx = np.argsort(feature_importance)
        importance_sorted = feature_importance[sorted_idx]
        gene_names_sorted = np.array(gene_list)[sorted_idx]

        ax = fig.add_subplot(1, 2 * len(class_map.items()), class_idx + 1)
        ax.barh(range(len(importance_sorted)), importance_sorted, color=colors, alpha=1)
        ax.set_title(class_name, fontsize=24)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_ylim([-0.8, len(gene_names_sorted)])
        ax.set_yticks(range(len(gene_names_sorted)))
        if '_' in gene_names_sorted[0]:
            ax.set_yticklabels([item.split('_')[1] for item in gene_names_sorted])
        else:
            ax.set_yticklabels(gene_names_sorted)
        ax.tick_params(axis='x', which='major', labelsize=18)

        if class_idx == 0:
            ax.set_ylabel('Attention weights only', fontsize=16)

    fig.tight_layout(pad=2)
    fig.savefig(os.path.join(os.path.dirname(embedding_save_path), 'feature_importance.png'))

    for class_idx, class_name in class_map.items():
        subject_indices = (y_true_arr == class_idx).flatten()

        attention_curr_class = feature_attention_arr[subject_indices, ...].mean(axis=0)
        mlp_weight_curr_class = mlp_weights[class_idx, :]
        attention_curr_class = attention_curr_class * mlp_weight_curr_class[None, :]
        attention_curr_class = (attention_curr_class - attention_curr_class.min()) / (
            attention_curr_class.max() - attention_curr_class.min())

        feature_importance = attention_curr_class.sum(axis=0)
        # Sort descendingly.
        sorted_idx = np.argsort(feature_importance)
        importance_sorted = feature_importance[sorted_idx]
        gene_names_sorted = np.array(gene_list)[sorted_idx]

        ax = fig.add_subplot(1, 2 * len(class_map.items()), len(class_map.items()) + class_idx + 1)
        ax.barh(range(len(importance_sorted)), importance_sorted, color=colors, alpha=1)
        ax.set_title(class_name, fontsize=24)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_ylim([-0.8, len(gene_names_sorted)])
        ax.set_yticks(range(len(gene_names_sorted)))
        if '_' in gene_names_sorted[0]:
            ax.set_yticklabels([item.split('_')[1] for item in gene_names_sorted])
        else:
            ax.set_yticklabels(gene_names_sorted)
        ax.tick_params(axis='x', which='major', labelsize=18)

        if class_idx == 0:
            ax.set_ylabel('Attention weights scaled by MLP weights', fontsize=16)

    fig.tight_layout(pad=2)
    fig.savefig(os.path.join(os.path.dirname(embedding_save_path), 'feature_importance.png'))
    plt.close(fig)

    return

def softmax(x):
# Subtract max for numerical stability
  e_x = np.exp(x - np.max(x))
  return e_x / e_x.sum(axis=-1, keepdims=True)

In [53]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the data.
train_loader, val_loader, test_loader, num_classes = prepare_dataloaders(args)

model = HypergraphScatteringNet(
    in_channels=64,
    hidden_channels=64,
    out_channels=num_classes,
    num_features=29,
    trainable_laziness=False,
    trainable_scales=True,
    activation=None,  # just get one layer of wavelet transform
    fixed_weights=True,
    layout=['hsm'],
    normalize='right',
    pooling='attention',
    scale_list=[0,1,2,4]
)
model.eval()
model.to(device)

model_save_path = os.path.join('/home/jcr222/workspace/hypergraphs/hypergraph-wavelets/results/mibi/dataset-mibi-patchified_all_genes_kHop-1_features-29_trainable_scales-False_seed-1/model.pt')
embedding_save_path = '/home/jcr222/workspace/hypergraphs/hypergraph-wavelets/results/mibi/dataset-mibi-patchified_all_genes_kHop-1_features-29_trainable_scales-False_seed-1'
attention_save_path = os.path.join(embedding_save_path,'attentions.npz')


In [54]:
model.eval()
model.load_state_dict(torch.load(model_save_path, map_location=device, weights_only=True))

<All keys matched successfully>

In [57]:
if not os.path.isfile(attention_save_path):
    save_test_set_attentions(model, test_loader, device, attention_save_path)

100%|██████████| 1070/1070 [02:00<00:00,  8.89it/s]


In [58]:
gene_list = test_loader.dataset.dataset.gene_list
class_map = test_loader.dataset.dataset.class_map
npzfile = np.load(attention_save_path,allow_pickle=True)


In [60]:
visualize_test_set_attention(attention_save_path, gene_list=gene_list, class_map=class_map)